<a href="https://colab.research.google.com/github/mayuresh-tungare/c3m2/blob/main/mini-1/Mini_Assignment_1_Mayuresh_Tungare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You are provided with a data set titled ‘Lung Cancer Patient Health and Treatment Records’, which captures detailed information about patient demographics, diagnosis stage, lifestyle risk factors, comorbidities and treatment outcomes.
Note: This data set was inspired by various healthcare-related learning resources and is not intended to represent real patient data.
With the given data set, solve the following tasks using PySpark.

In [2]:
from pyspark.sql import SparkSession
import requests
import os

# Create a SparkSession
spark = SparkSession.builder.appName("LungCancerDataLoading").getOrCreate()

# URL of the raw CSV file on GitHub
csv_url = "https://raw.githubusercontent.com/mayuresh-tungare/c3m2/main/mini-1/Lung%20Cancer.csv"

# Define a local path to save the CSV file
local_csv_path = "/tmp/Lung Cancer.csv"

# Download the CSV file
response = requests.get(csv_url)
response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
with open(local_csv_path, 'wb') as f:
    f.write(response.content)

# Load the dataset from the local path
df = spark.read.csv(local_csv_path, header=True, inferSchema=True)

# Display the schema and a few rows to verify
df.printSchema()
df.show(5)

root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: string (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking

Task 1: Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format.

In [23]:
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, DateType
from pyspark.sql.functions import col, when

def clean_data(df):
    """
    Removes duplicates, ensures correct data types, and converts yes/no fields to 1/0.
    """
    # Remove duplicate rows
    df = df.dropDuplicates()

    # Convert yes/no columns to 1/0
    # Only include columns that are actually expected to contain 'yes' or 'no' values
    yes_no_columns = ['family_history'] # Corrected: specify only actual yes/no columns
    for column in yes_no_columns:
        if column in df.columns:
            df = df.withColumn(column,
                              when(col(column).isin(['yes', 'Yes', 'YES']), 1)
                              .when(col(column).isin(['no', 'No', 'NO']), 0)
                              .otherwise(col(column)))

    # Cast numerical columns to appropriate types
    numerical_columns = ['age', 'bmi', 'treatment_duration_days']
    for column in numerical_columns:
        if column in df.columns:
            df = df.withColumn(column, col(column).cast(DoubleType()))

    # Cast date columns to DateType
    date_columns = ['diagnosis_date', 'treatment_end_date']
    for column in date_columns:
        if column in df.columns:
            df = df.withColumn(column, col(column).cast(DateType()))

    return df
clean_data(df).show()

+----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  id| age|gender|       country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+----+----+------+--------------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  72|54.0|Female|Czech Republic|    2017-06-30|    Stage IV|             0|Passive Smoker|24.0|              160|           1|     0|        0|           1|       Surgery|        2019-04-06|       0|
| 190|34.0|  Male|       Estonia|    2015-02-13|     Stage I|             1| Former Smoker|38.8|              252|           1|     1|        1|           0|      Combined|        2016-12-18|     

Task 2: Write a function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. Then, return the average treatment duration for each treatment type.

In [21]:
from pyspark.sql.functions import datediff, avg, col

def treatment_duration_analysis(df):
    """
    Adds treatment_duration_days column and returns average treatment duration by treatment type.
    """
    # Add treatment_duration_days column
    df = df.withColumn('treatment_duration_days',
                       datediff(col('end_treatment_date'), col('diagnosis_date')))

    # Calculate average treatment duration for each treatment type
    avg_duration_by_treatment = df.groupBy('treatment_type').agg(
        avg('treatment_duration_days').alias('avg_treatment_duration')
    )

    return avg_duration_by_treatment
treatment_duration_analysis(df).show()

+--------------+----------------------+
|treatment_type|avg_treatment_duration|
+--------------+----------------------+
|     Radiation|    458.40320462900917|
|  Chemotherapy|    458.39540091909953|
|      Combined|     457.8152186120058|
|       Surgery|    457.73744630723684|
+--------------+----------------------+



Task 3: Write a function that returns the smoking_status group with the highest survival rate.  

In [20]:
from pyspark.sql.functions import col, sum, count

def highest_survival_smoking_group(df):
    """
    Returns the smoking_status group with the highest survival rate.
    """
    # Calculate survival rate for each smoking_status group
    survival_rates = df.groupBy('smoking_status').agg(
        (sum(col('survived')) / count(col('survived'))).alias('survival_rate')
    )

    # Find the group with the highest survival rate
    highest_survival_group = survival_rates.orderBy(col('survival_rate').desc()).first()

    return highest_survival_group['smoking_status']
print(highest_survival_smoking_group(df))

Never Smoked


Task 4: Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV.  

In [18]:
from pyspark.sql.functions import col, sum, count, desc

def top_three_stage_iv_countries(df):
    """
    Returns the top three countries with the highest percentage of patients diagnosed in Stage IV.
    """
    # Calculate total patients and Stage IV patients by country
    stage_iv_percentage = df.groupBy('country').agg(
        (sum(when(col('cancer_stage') == 'Stage IV', 1).otherwise(0)) / count(col('country'))).alias('stage_iv_percentage')
    )

    # Get top 3 countries with highest Stage IV percentage
    top_three = stage_iv_percentage.orderBy(col('stage_iv_percentage').desc()).limit(3)

    return top_three

# Call the function and display its content
top_three_stage_iv_countries(df).show()

+--------------+-------------------+
|       country|stage_iv_percentage|
+--------------+-------------------+
|        Greece| 0.2550223889628464|
|       Croatia| 0.2542700223308588|
|Czech Republic|0.25291166185190816|
+--------------+-------------------+



Task 5: Write a function that filters patients who:  

Are male  

Diagnosed in Stage III or IV  

Have a family history of cancer  

Are current smokers  

Have a BMI > 30  

Survived

Return the average age and the percentage of these patients who had hypertension.

In [28]:
from pyspark.sql.functions import col, avg, sum, count, when

def filter_and_analyze_patients(df):
    """
    Filters patients based on criteria and returns average age and hypertension percentage.
    """
    # Filter patients based on all criteria
    filtered_df = df.filter(
        (col('gender') == 'Male') &
        (col('cancer_stage').isin(['Stage III', 'Stage IV'])) & # Corrected: diagnosis_stage to cancer_stage
        (col('family_history') == 1) &                          # Corrected: family_history_cancer to family_history
        (col('smoking_status') == 'Current Smoker') &           # Corrected: 'Current' to 'Current Smoker'
        (col('bmi') > 30) &
        (col('survived') == 1)
    )

    # Calculate average age and hypertension percentage
    result = filtered_df.agg(
        avg('age').alias('average_age'),
        (sum(when(col('hypertension') == 1, 1).otherwise(0)) / count('*')).alias('hypertension_percentage')
    )

    return result

# Ensure the DataFrame is cleaned before filtering
cleaned_df = clean_data(df)
filter_and_analyze_patients(cleaned_df).show()

+------------------+-----------------------+
|       average_age|hypertension_percentage|
+------------------+-----------------------+
|55.179398872886665|     0.7476518472135254|
+------------------+-----------------------+

